**Crime and Enrichment Data Pipeline**

Package Imports

In [1]:
import pandas as pd
import os

Ingestion Layer (Crime Data)

In [2]:
basePath = '6YearSpanDatasets'

# This ingestion method loads each individual force's monthly data one by one, appending it into an array and then adding it to a dataframe

# List of forces that are going to be analysed
forces = ['metropolitan', 'thames', 'surrey', 'sussex']

# Array which is cumulatively appended for each force's crime data for each month between January 2019 to December 2024
allData = []

# Checks each folder in the updatedDatasets folder
for folder in os.listdir(basePath):
    # Concatenates the folder name to the updatedDatasets to create a new file path
    filePath = os.path.join(basePath, folder)
    # Checks if the folder actually exists
    if os.path.isdir(filePath):
        # Checks each file in the file path
        for file in os.listdir(filePath):
            # Checks the file contains one of the police forces in the filename
            if any(force in file.lower() for force in forces):
                # Updates file path to include the folder
               file_path = os.path.join(filePath, file)
               
               df=pd.read_csv(file_path)
               df['Month'] = folder
               allData.append(df)
    
# If statement to check the ingestion layer correctly loaded data
if allData:
    masterDf = pd.concat(allData, ignore_index=True)
    print(f"Ingestion Layer Record Check: Raw crime records: {len(masterDf)}")
else:
    print("No data found to process")


Ingestion Layer Record Check: Raw crime records: 9453125


Initial Data Inspection

In [3]:
# Shows number of records in the master data
print(f"Before preprocessing, there are {len(masterDf)} values in the data")

# Shows each column and if they have any null values
print(f"\nList of null values in every column \n {masterDf.isnull().sum()}\n")

# Shows the unique values for each column
for col in masterDf:
    print(f" - The unique values for {col} are {masterDf[col].unique()}")


Before preprocessing, there are 9453125 values in the data

List of null values in every column 
 Crime ID                 2111038
Month                          0
Reported by                    0
Falls within                   0
Longitude                 176614
Latitude                  176614
Location                       0
LSOA code                 176644
LSOA name                 176644
Crime type                     0
Last outcome category    2111038
Context                  9453125
dtype: int64

 - The unique values for Crime ID are ['a8977a2a4e14252420371eb993d52e4d0b8288a7c833e66ba79592b3e2b982e7' nan
 '934e173f2bc2e1dd3a257b37939d8f97575d3eeb89ff0c3eee2c73f16630edc5' ...
 'cb80080e0164fa9d12767bcfe912ed902b1bc3db0ce83d50b7f6bc055d313392'
 'c49ac623430308ffb6f815e12725a9d3244c7bf3275592a22953ed4f5964c326'
 '14fb64f2b5de925c7c1cec2234f64d221dada2af05b6bc07fe41db9139e021ea']
 - The unique values for Month are ['2019-01' '2019-02' '2019-03' '2019-04' '2019-05' '2019-06' '2019-07'

Deduplication

In [4]:
# Checks the number of total records in the crime data before any transformation or cleaning
totalBeforeDupRemoval = len(masterDf)
print(f"There are {totalBeforeDupRemoval} values in the dataset before any processing at all")

# Checks the number of duplicate records in the crime data
duplicateCount = masterDf.duplicated().sum()
print(f"There are {duplicateCount} duplicate rows.")

# Drops the duplicate records
masterDf = masterDf.drop_duplicates()

#
totalAfterDupRemoval = len(masterDf)
print(f"There are {totalAfterDupRemoval} values in the dataset after removing duplicates")

There are 9453125 values in the dataset before any processing at all
There are 1011718 duplicate rows.
There are 8441407 values in the dataset after removing duplicates


Null Value Handling

In [5]:
# From cell 4 there are missing values in Crime ID, Longitude, Latitude, LSOA Code, LSOA Name, Last outcome category and Context

# Shows each column and if they have any null values
print(f"\nList of null values in every column before handling null values\n {masterDf.isnull().sum()}\n")

# In the data, Crime ID, Longitude, Latitude, LSOA Code, LSOA Name, Last outcome category and Context contained null values

# Context was empty for every single record so can be dropped
masterDf = masterDf.drop("Context", axis='columns')

# For remaining columns, too much crime data would be lost if dropped so placeholders were assigned instead

# Crime ID: add placeholder value "No Crime ID"
# Longitude and Latitude: add placeholder value 0
# LSOA code: add placeholder value "No LSOA Code Assigned"
# LSOA name: add placeholder value "No LSOA name Assigned"
# Last outcome category: add placeholder value "Unknown outcome category"
# Context: drop since every record has null values in context

placeholders = {
    'Crime ID': 'No Crime ID',
    'Longitude': 0,
    'Latitude' : 0,
    'LSOA code' : 'No LSOA Code assigned',
    'LSOA name' : 'No LSOA Name assigned',
    'Last outcome category' : 'Unknown outcome category'
}

masterDf = masterDf.fillna(value=placeholders)

# Shows each column and if they have any null values
print(f"\nList of null values in every column \n {masterDf.isnull().sum()}\n")


List of null values in every column before handling null values
 Crime ID                 1155292
Month                          0
Reported by                    0
Falls within                   0
Longitude                 167326
Latitude                  167326
Location                       0
LSOA code                 167356
LSOA name                 167356
Crime type                     0
Last outcome category    1155292
Context                  8441407
dtype: int64


List of null values in every column 
 Crime ID                 0
Month                    0
Reported by              0
Falls within             0
Longitude                0
Latitude                 0
Location                 0
LSOA code                0
LSOA name                0
Crime type               0
Last outcome category    0
dtype: int64



Column Handling

In [6]:
# Dropping "Falls within" column as each record mirrored the value within the "Reported by" column
masterDf = masterDf.drop("Falls within", axis='columns')

Text Standardisation

In [7]:
# Clean Force names to be just the name of the police force without "Force" or other keywords
masterDf['Force'] = masterDf['Reported by'].str.replace("Police", "").str.strip()

# Clean metropolitan
masterDf.loc[masterDf['Force'].str.contains('Metropolitan', case=False), 'Force'] = 'Metropolitan'
masterDf['Force'] = masterDf['Force'].str.title()

# Standardize 'Crime type' and 'Last outcome category'
masterDf['Crime type'] = masterDf['Crime type'].str.strip().str.capitalize()
masterDf['Last outcome category'] = masterDf['Last outcome category'].str.strip().str.capitalize()

# Dropped Reported by column since a cleaner, standardised Force column was derived from it
masterDf = masterDf.drop('Reported by', axis='columns')

# For loop to ensure that the column was definitely dropped
for col in masterDf:
    print(col)
    
print(f"\n{masterDf.head()}")

Crime ID
Month
Longitude
Latitude
Location
LSOA code
LSOA name
Crime type
Last outcome category
Force

                                            Crime ID    Month  Longitude  \
0  a8977a2a4e14252420371eb993d52e4d0b8288a7c833e6...  2019-01  -0.709911   
1                                        No Crime ID  2019-01   0.140192   
2  934e173f2bc2e1dd3a257b37939d8f97575d3eeb89ff0c...  2019-01   0.140192   
3  4f5b7e424bc78b1fb8c32e07da61176d2cbc5a3849d8e1...  2019-01   0.140634   
4  53d960600a4a9f54b785f598af4c75bdef2f79bce1a41b...  2019-01   0.141143   

    Latitude                     Location  LSOA code  \
0  50.784615     On or near Rochester Way  E01031384   
1  51.582311       On or near Hatch Grove  E01000027   
2  51.582311       On or near Hatch Grove  E01000027   
3  51.583427        On or near Rams Grove  E01000027   
4  51.590873  On or near Furze Farm Close  E01000027   

                   LSOA name                    Crime type  \
0                  Arun 018E  Violence an

Year Conversion 

In [8]:
# Converting the existing time data into a readable pandas time attribute for future analysis
yearConversion = pd.to_datetime(masterDf['Month'])
masterDf['Year'] = yearConversion.dt.year

Crime Data Aggregation

In [9]:
print(f"Pre Crime Data Aggregation Record Check: Crime records: {len(masterDf)}")

# Creating the aggregations, grouping by the defined grain (Force X Year X LSOA name x Crime Type)
crimeAggregation = masterDf.groupby(['Force', 'Year', 'LSOA name', 'Crime type']).agg(
    crimeCount=('Crime ID', 'count'),
    Latitude=('Latitude', 'mean'),
    Longitude=('Longitude', 'mean')
).reset_index()

crimeAggregation = crimeAggregation.rename(columns={'crimeCount': 'Crime count'})

crimeAggregation.head(10)

print(f"Post Crime Data Aggregation Record Check: Crime records: {len(crimeAggregation)}")

Pre Crime Data Aggregation Record Check: Crime records: 8441407
Post Crime Data Aggregation Record Check: Crime records: 580912


Ingestion Layer (ONS Population Data)

In [10]:
xls = pd.ExcelFile('policeforceareas1991to2024.xlsx')
# Extracting only the Mid-2011 to Mid-2020 and Mid-2021 to Mid-2024 sheet as these contain the years that are to be analysed
pop_11_20 = pd.read_excel(xls, 'Mid-2011 to Mid-2020', skiprows=3)
pop_21_24 = pd.read_excel(xls, 'Mid-2021 to Mid-2024', skiprows=3)

# Concatenating the data together to form one raw data set of the ONS Population
popAll = pd.concat([pop_11_20, pop_21_24], axis=0)
popAll.head(12)

print(f"Population Data Ingestion Layer Record Check: Raw population records: {len(popAll)}")




Population Data Ingestion Layer Record Check: Raw population records: 602


Feature Engineering

In [11]:
# Calculates the total population for a force for a given year
populationColumns = popAll.columns[3:]
popAll['Population'] = popAll[populationColumns].sum(axis=1)

popClean = popAll[['PFA 2023 Name', 'Year', 'Population']].copy()
popClean.columns = ['Force', 'Year', 'Population']
popClean = popClean.reset_index(drop=True)

print(popClean.head())

                 Force  Year  Population
0  Metropolitan Police  2011     8196995
1  Metropolitan Police  2012     8313253
2  Metropolitan Police  2013     8431450
3  Metropolitan Police  2014     8539625
4  Metropolitan Police  2015     8651877


Text Standardisation

In [12]:
popClean['Force'] = popClean['Force'].str.replace("Police", "").str.strip()


Population Data Aggregation


In [ ]:
forceYearTotals = masterDf.groupby(['Force', 'Year']).size().reset_index(name='Total Force Crime Count')

assert forceYearTotals['Force'].notnull().all(), "ERROR: 'Force contains null values in aggregated crime data"
assert forceYearTotals['Year'].notnull().all(), "ERROR: 'Year contains null values in aggregated crime data"
assert popClean['Force'].notnull().all(), "ERROR: 'Force contains null values in aggregated population data"
assert popClean['Force'].notnull().all(), "ERROR: 'Force contains null values in aggregated population data"

forceRatesBaseline = pd.merge(forceYearTotals, popClean, on=['Force', 'Year'], how='left')

# Validation Check for the matching rate of the crime data to the population data
matchedPopRows = forceRatesBaseline['Population'].notnull().sum()
totalBaselineRows = len(forceRatesBaseline)
popMatchRate = (matchedPopRows/totalBaselineRows) * 100
print(f"The matching rate of force data to population data is {popMatchRate:.2f}% with {matchedPopRows} rows matching out of {totalBaselineRows}")
assert popMatchRate == 100.0, "ERROR: Some force/year combinations failed to match the population baseline"

forceRatesBaseline['Force Crimes per 1000'] = (forceRatesBaseline['Total Force Crime Count'] / forceRatesBaseline['Population']) * 1000

The matching rate of force data to population data is 100.00 with 24 rows matching out of 24


Ingestion Layer (Index of Deprivation Data)

In [14]:
# Dictionary for the LSOA names that are covered by the chosen police forces
force_lookup = {
    # Metropolitan 
    'Barking and Dagenham': 'Metropolitan', 'Barnet': 'Metropolitan', 'Bexley': 'Metropolitan',
    'Brent': 'Metropolitan', 'Bromley': 'Metropolitan', 'Camden': 'Metropolitan',
    'Croydon': 'Metropolitan', 'Ealing': 'Metropolitan', 'Enfield': 'Metropolitan',
    'Greenwich': 'Metropolitan', 'Hackney': 'Metropolitan', 'Hammersmith and Fulham': 'Metropolitan',
    'Haringey': 'Metropolitan', 'Harrow': 'Metropolitan', 'Havering': 'Metropolitan',
    'Hillingdon': 'Metropolitan', 'Hounslow': 'Metropolitan', 'Islington': 'Metropolitan',
    'Kensington and Chelsea': 'Metropolitan', 'Kingston upon Thames': 'Metropolitan',
    'Lambeth': 'Metropolitan', 'Lewisham': 'Metropolitan', 'Merton': 'Metropolitan',
    'Newham': 'Metropolitan', 'Redbridge': 'Metropolitan', 'Richmond upon Thames': 'Metropolitan',
    'Southwark': 'Metropolitan', 'Sutton': 'Metropolitan', 'Tower Hamlets': 'Metropolitan',
    'Waltham Forest': 'Metropolitan', 'Wandsworth': 'Metropolitan', 'Westminster': 'Metropolitan',
    'City of London': 'Metropolitan', 
    # Surrey
    'Elmbridge': 'Surrey', 'Epsom and Ewell': 'Surrey', 'Guildford': 'Surrey',
    'Mole Valley': 'Surrey', 'Reigate and Banstead': 'Surrey', 'Runnymede': 'Surrey',
    'Spelthorne': 'Surrey', 'Surrey Heath': 'Surrey', 'Tandridge': 'Surrey',
    'Waverley': 'Surrey', 'Woking': 'Surrey',

    # Sussex
    'Adur': 'Sussex', 'Arun': 'Sussex', 'Brighton and Hove': 'Sussex',
    'Chichester': 'Sussex', 'Crawley': 'Sussex', 'Eastbourne': 'Sussex',
    'Hastings': 'Sussex', 'Horsham': 'Sussex', 'Lewes': 'Sussex',
    'Mid Sussex': 'Sussex', 'Rother': 'Sussex', 'Wealden': 'Sussex', 'Worthing': 'Sussex',

    # Thames Valley
    'Bracknell Forest': 'Thames Valley', 'Buckinghamshire': 'Thames Valley', 'Cherwell': 'Thames Valley',
    'Milton Keynes': 'Thames Valley', 'Oxford': 'Thames Valley', 'Reading': 'Thames Valley',
    'Slough': 'Thames Valley', 'South Oxfordshire': 'Thames Valley', 'Vale of White Horse': 'Thames Valley',
    'West Berkshire': 'Thames Valley', 'West Oxfordshire': 'Thames Valley',
    'Windsor and Maidenhead': 'Thames Valley', 'Wokingham': 'Thames Valley'
}


iod2025 = pd.read_csv("File_7_IoD2025_All_Ranks_Scores_Deciles_Population_Denominators.csv")
print(f"Deprivation Data Ingestion Layer Record Check: Raw population records: {len(iod2025)}")
# Mapping the LDOA names to the force mapping above where applicable
assert iod2025['Local Authority District name (2024)'].notnull().all(), "ERROR: Raw IoD contains nulls in join key column"
iod2025['Force'] = iod2025['Local Authority District name (2024)'].map(force_lookup)

# Validation Check for the matching rate of the local authority to force mapping
targetForces = ['Metropolitan', 'Thames Valley', 'Surrey', 'Sussex']
matchCount = iod2025['Force'].isin(targetForces).sum()
totalRows = len(iod2025)
matchRate = (matchCount/totalRows) * 100
print(f"The matching rate of IoD Local Authority to Force is {matchRate:.2f} with {matchCount} rows matching out of {totalRows}")

# Showing how many records are not in covered by the 4 forces
excludedEntries = iod2025['Force'].isna().sum()
print(f"There are {excludedEntries} out of {len(iod2025)} LSOAs that are outside the 4 forces")

Deprivation Data Ingestion Layer Record Check: Raw population records: 33755
The matching rate of IoD Local Authority to Force is 24.37 with 8227 rows matching out of 33755
There are 25528 out of 33755 LSOAs that are outside the 4 forces


Feature Engineering

In [15]:
# Adding column to flag if a neighbourhood is part of the top 10% of deprived areas
iodFiltered = iod2025.dropna(subset=['Force']).copy()
iodFiltered['is_decile_1'] = iodFiltered['Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOAs)'] == 1

print(iodFiltered['is_decile_1'].value_counts())

is_decile_1
False    7968
True      259
Name: count, dtype: int64


Aggregation (Index of Deprivation Data)

In [16]:
imd_final = iodFiltered.groupby('Force').agg(
    # Average Index of Deprivation Score per force
    AvgIMDScore=('Index of Multiple Deprivation (IMD) Score', 'mean'),
    # Average Income decile score per force
    AvgIncomeDecile=('Income Decile (where 1 is most deprived 10% of LSOAs)', 'mean'),
    # Number of neighbourhoods that force spans
    NeighbourhoodCount=('LSOA code (2021)', 'count'),
    # Number of neighbours in the force that are in the top 10 percent of deprivation
    HighlyDeprivedCount=('is_decile_1', 'sum')
    
).reset_index()

# Percentage of neighbourhoods in the force trhat are in the top 10 percent of deprivation
imd_final['HighlyDeprivedPercentage'] = (imd_final['HighlyDeprivedCount'] / imd_final['NeighbourhoodCount']) * 100

display(imd_final)

,Force,AvgIMDScore,AvgIncomeDecile,NeighbourhoodCount,HighlyDeprivedCount,HighlyDeprivedPercentage
0,Metropolitan,22.128675,4.543052,4994,201,4.024830
1,Surrey,10.108495,7.528512,719,0,0.000000
2,Sussex,17.677184,6.051506,1029,48,4.664723
3,Thames Valley,12.673253,6.952189,1485,10,0.673401


Final merging of all the data

In [17]:
baseCount = len(crimeAggregation)
print(f"Pre merge 1 record check: Number of records: {baseCount}")
finalData = pd.merge(crimeAggregation, forceRatesBaseline, on=['Force', 'Year'], how='left')
print(f"Post merge 1 record check: Number of records: {len(finalData)}")
assert len(finalData) == baseCount, "ERROR: Row inflation detected after population data merge"
completeData = pd.merge(finalData, imd_final, on=['Force'], how='left')
print(f"Post merge 2 record check: Number of records: {len(completeData)}")
assert len(completeData) == baseCount, "ERROR: Row inflation detected after deprivation data merge"

display(completeData)

Pre merge 1 record check: Number of records: 580912
Post merge 1 record check: Number of records: 580912
Post merge 2 record check: Number of records: 580912


,Force,Year,LSOA name,Crime type,Crime count,Latitude,Longitude,Total Force Crime Count,Population,Force Crimes per 1000,AvgIMDScore,AvgIncomeDecile,NeighbourhoodCount,HighlyDeprivedCount,HighlyDeprivedPercentage
0,Metropolitan,2019,Adur 006A,Other theft,1,50.837110,-0.335092,980566,8880978,110.411939,22.128675,4.543052,4994,201,4.024830
1,Metropolitan,2019,Amber Valley 016A,Violence and sexual offences,1,52.982886,-1.493708,980566,8880978,110.411939,22.128675,4.543052,4994,201,4.024830
2,Metropolitan,2019,Arun 009F,Violence and sexual offences,1,50.816571,-0.543651,980566,8880978,110.411939,22.128675,4.543052,4994,201,4.024830
3,Metropolitan,2019,Arun 017A,Other crime,1,50.785967,-0.662250,980566,8880978,110.411939,22.128675,4.543052,4994,201,4.024830
4,Metropolitan,2019,Arun 017A,Violence and sexual offences,1,50.785967,-0.662250,980566,8880978,110.411939,22.128675,4.543052,4994,201,4.024830
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
580907,Thames Valley,2024,Wokingham 020E,Other crime,2,51.374259,-0.802241,194795,2640201,73.780367,12.673253,6.952189,1485,10,0.673401
580908,Thames Valley,2024,Wokingham 020E,Other theft,1,51.375766,-0.804458,194795,2640201,73.780367,12.673253,6.952189,1485,10,0.673401
580909,Thames Valley,2024,Wokingham 020E,Public order,1,51.375957,-0.800300,194795,2640201,73.780367,12.673253,6.952189,1485,10,0.673401
580910,Thames Valley,2024,Wokingham 020E,Vehicle crime,3,51.371635,-0.800375,194795,2640201,73.780367,12.673253,6.952189,1485,10,0.673401


Final Validation Checks

In [18]:
# Check the number of duplicates in the final data
grainDuplicates = completeData.duplicated(subset=['Force', 'Year', 'LSOA name', 'Crime type']).sum()
print(f"Validation - Duplicate records at reporting grain: {grainDuplicates}")

# Check for number of null values in the data
print("\nFinal Missing Value Summary:")
print(completeData[['Force', 'Year', 'LSOA name', 'Crime type']].isnull().sum())

# General summary of final dataset statistics
print(f"\n--- Final Dataset Summary Statistics ---")
print(f"Total Grain Rows: {len(completeData)}")
print(f"Unique Police Forces Covered: {completeData['Force'].nunique()} ({completeData['Force'].unique()})")
print(f"Temporal Window Span: {completeData['Year'].min()} to {completeData['Year'].max()}")







Validation - Duplicate records at reporting grain: 0

Final Missing Value Summary:
Force         0
Year          0
LSOA name     0
Crime type    0
dtype: int64

--- Final Dataset Summary Statistics ---
Total Grain Rows: 580912
Unique Police Forces Covered: 4 (['Metropolitan' 'Surrey' 'Sussex' 'Thames Valley'])
Temporal Window Span: 2019 to 2024


Dataset Exporting

In [19]:
completeData.to_csv('processedAndAggregatedCrimeData.csv', index=False)
print("\nSuccess: BI-ready reporting dataset successfully exported.")


Success: BI-ready reporting dataset successfully exported.
